# Tsurugi NNUE training on self-play data (Kaggle GPU)

Trains the 768 -> 2x256 -> 1 network only on positions from Tsurugi's own self-play (`tsurugi datagen`), with
no Stockfish labels: the target blends the engine's search score with the game result (`WDL`).
Needs the self-play files attached as a dataset (`datagen-*.txt` or `.txt.gz` under `/kaggle/input`), **Accelerator: GPU**.
The network appears under **Output** as `tsurugi-v3.nnue` (+ `tsurugi-v3.json` training log).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf tsurugi && git clone --depth 1 https://github.com/Abhayindia/tsurugi.git

In [ ]:
import glob
NAME, WDL, EPOCHS = "tsurugi-v3", 0.25, 20
files = sorted(f for ext in ("txt", "txt.gz") for f in glob.glob(f"/kaggle/input/**/datagen-*.{ext}", recursive=True))
print(len(files), "files"); assert files
!python tsurugi/scripts/nnue/prepare_selfplay.py --out /tmp/sp {" ".join(files)}

In [ ]:
!python tsurugi/scripts/nnue/train.py --data /tmp/sp --out /kaggle/working/{NAME}.nnue --epochs {EPOCHS} --wdl {WDL} --batch 16384 --threads 4

In [ ]:
# Build the engine and check the network gives exactly the engine's evaluation.
!cd tsurugi && cmake -S . -B build -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build build -j4 --target tsurugi > /dev/null && echo built
!pip install -q chess
!python tsurugi/scripts/nnue/check_net.py tsurugi/build/tsurugi /kaggle/working/{NAME}.nnue
!cp /tmp/sp/meta.json /kaggle/working/{NAME}-data.json && ls -la /kaggle/working